### Pip install que precisam ocorrer antes de importe de blibliotecas especificas

In [3]:
#pip install torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu121
pip install "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"
pip install --no-deps trl peft accelerate bitsandbytes
pip install pandas scikit-learn google-generativeai matplotlib ipywidgets

SyntaxError: invalid syntax (4246858852.py, line 2)

In [ ]:
# Imports de bibliotecas necessárias
import io
import os
import torch
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import ipywidgets as widgets
from typing import List
from datasets import Dataset
from sklearn.model_selection import train_test_split
from unsloth import FastLanguageModel
from trl import SFTTrainer, SFTConfig
import google.generativeai as genai

# Verificar disponibilidade da GPU
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Executando em: {device} | Dispositivo: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU'}")

In [ ]:
# def carregar_csv(paths_csv: List[str]) -> pd.DataFrame:
#     lista_dfs = []
#     encodings = ['utf-8', 'latin1', 'ISO-8859-1', 'cp1252']
#     seps = [';', ',']

#     for path in paths_csv:
#         df_carregado = None
#         for encoding in encodings:
#             for sep in seps:
#                 try:
#                     curr_df = pd.read_csv(path, sep=sep, encoding=encoding)
#                     if not curr_df.empty and 'PROPOSTA' in curr_df.columns:
#                         df_carregado = curr_df
#                         break
#                 except Exception:
#                     continue
#             if df_carregado is not None:
#                 lista_dfs.append(df_carregado)
#                 break

#     if lista_dfs:
#         return pd.concat(lista_dfs, ignore_index=True)
#     return pd.DataFrame()

# # Insira o caminho local exato do seu arquivo
# file_path = ["./com_propostas.csv"]
# df = carregar_csv(file_path)

# if df.empty:
#     raise ValueError("O DataFrame está vazio. Verifique o caminho e formato do arquivo com_propostas.csv.")

# print(f"Arquivo carregado com sucesso! Total de registros: {len(df)}")

In [ ]:
import os
import google.generativeai as genai
from datasets import Dataset
import pandas as pd
from sklearn.model_selection import train_test_split


def carregar_e_preparar_dados(caminho_csv: str) -> pd.DataFrame:
    # Carrega mantendo os nomes originais de colunas
    df = pd.read_csv(caminho_csv, sep=";", encoding="utf-8")
    df.columns = df.columns.str.strip().str.upper()

    # Define label verdadeiro para os dados reais
    df["LABEL_CATEGORY"] = "true"
    return df


def gerar_proposta_falsa(candidato: str, proposta_real: str) -> str:
    api_key = os.getenv("GOOGLE_API_KEY")
    if not api_key:
        return ""

    genai.configure(api_key=api_key)
    model = genai.GenerativeModel("gemini-1.5-flash")

    prompt = f"""Você é um gerador de dados sintéticos para treinamento de checagem de fatos.
Candidato: {candidato}
Proposta Real: {proposta_real}

Gere UMA proposta FALSA/DISTORCIDA que pareça ter sido dita pelo candidato "{candidato}", baseando-se na proposta real acima.
Aplique um exagero inviável, alteração de público-alvo ou inclusão de custos/regras absurdas.
Retorne APENAS o texto da proposta falsa."""

    try:
        response = model.generate_content(prompt)
        return response.text.strip() if response.text else ""
    except Exception as e:
        print(f"Erro ao gerar fake para {candidato}: {e}")
        return ""


# 1. Carregamento da base real
df_real = carregar_e_preparar_dados("./com_propostas.csv")

# 2. Geração sintética pareada por candidato
falsas_registros = []
print("Gerando propostas falsas vinculadas aos candidatos...")

for _, row in df_real.iterrows():
    cand_nome = row.get("NM_CANDIDATO", row.get("NM_URNA_CANDIDATO", "Desconhecido"))
    prop_real = row.get("PROPOSTA", "")

    if pd.isna(prop_real) or not str(prop_real).strip():
        continue

    prop_falsa = gerar_proposta_falsa(str(cand_nome), str(prop_real))

    if prop_falsa:
        falsas_registros.append(
            {
                "DS_CARGO": row.get("DS_CARGO", ""),
                "NM_UE": row.get("NM_UE", ""),
                "SQ_CANDIDATO": row.get("SQ_CANDIDATO", ""),
                "NM_CANDIDATO": row.get("NM_CANDIDATO", cand_nome),
                "NM_URNA_CANDIDATO": row.get("NM_URNA_CANDIDATO", cand_nome),
                "PROPOSTA": prop_falsa,
                "LABEL_CATEGORY": "false",
            }
        )

df_falsas = pd.DataFrame(falsas_registros)
df_final = pd.concat([df_real, df_falsas], ignore_index=True)

# Embaralha os dados para treino
df_final = df_final.sample(frac=1, random_state=42).reset_index(drop=True)

In [ ]:
max_seq_length = 8192
lora_rank = 16          # Rank 16 otimiza o uso da GPU sem perda relevante de precisão

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = "unsloth/mistral-7b-instruct-v0.3-bnb-4bit",
    max_seq_length = max_seq_length,
    load_in_4bit = True,
    gpu_memory_utilization = 0.85, # Reserva espaço para prevenção de Out-Of-Memory
)

# Configuração LoRA/PEFT para SFT
model = FastLanguageModel.get_peft_model(
    model,
    r = lora_rank,
    target_modules = ["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
    lora_alpha = lora_rank,
    lora_dropout = 0,
    bias = "none",
    use_gradient_checkpointing = "unsloth", # Crucial para VRAM de 12GB
    random_state = 3407,
)

In [ ]:
# Template ajustado para incluir a relação Candidato + Proposta
prompt_template = """### Instrução:
Classifique a proposta política a seguir atribuída ao candidato {candidato} como 'verdadeira' (true) ou 'falsa' (false).

### Candidato:
{candidato}

### Proposta:
{proposta}

### Resposta:
{resposta}"""


def formatar_prompts(dataframe):
    textos = []
    for _, row in dataframe.iterrows():
        candidato = row["NM_CANDIDATO"]
        if pd.isna(candidato) or not str(candidato).strip():
            candidato = row["NM_URNA_CANDIDATO"]

        texto = (
            prompt_template.format(
                candidato=candidato,
                proposta=row["PROPOSTA"],
                resposta=row["LABEL_CATEGORY"],
            )
            + tokenizer.eos_token
        )
        textos.append(texto)
    return pd.DataFrame({"text": textos})


# Divisão de Treino e Validação (80/20)
train_df, val_df = train_test_split(
    df_final, test_size=0.2, random_state=42, stratify=df_final["LABEL_CATEGORY"]
)

dataset_treino = Dataset.from_pandas(formatar_prompts(train_df))
dataset_validacao = Dataset.from_pandas(formatar_prompts(val_df))

In [ ]:
trainer = SFTTrainer(
    model = model,
    tokenizer = tokenizer,
    train_dataset = dataset_treino,
    eval_dataset = dataset_validacao,
    dataset_text_field = "text",
    max_seq_length = max_seq_length,
    dataset_num_proc = 2,
    packing = False, # Define True se quiser acelerar sequências curtas
    args = SFTConfig(
        per_device_train_batch_size = 2,     # Ajustado para 12GB VRAM
        gradient_accumulation_steps = 4,     # Simula batch size efetivo de 8
        warmup_ratio = 0.05,
        max_steps = 120,                     # Modifique para num_train_epochs se preferir
        learning_rate = 2e-4,
        fp16 = not torch.cuda.is_bf16_supported(),
        bf16 = torch.cuda.is_bf16_supported(), # Suportado pela arquitetura Ampere da RTX 3060
        logging_steps = 5,
        optim = "adamw_8bit",                # Reduz consumo de RAM/VRAM nos otimizadores
        weight_decay = 0.01,
        lr_scheduler_type = "linear",
        seed = 3407,
        output_dir = "outputs_propostas",
    ),
)

# Iniciar Treinamento SFT
trainer_stats = trainer.train()
print("Treinamento finalizado com sucesso!")

In [ ]:
# # Ativar modo rápido de inferência do Unsloth
# FastLanguageModel.for_inference(model)

# def classificar_proposta(texto_proposta: str) -> str:
#     prompt = prompt_template.format(texto_proposta, "")
#     inputs = tokenizer([prompt], return_tensors = "pt").to("cuda")
    
#     outputs = model.generate(
#         **inputs, 
#         max_new_tokens = 10, 
#         use_cache = True,
#         temperature = 0.1
#     )
#     resposta = tokenizer.batch_decode(outputs)
#     return resposta[0].split("### Resposta:\n")[-1].replace(tokenizer.eos_token, "").strip()

# # Teste com novos exemplos
# exemplos = [
#     "Criar programa municipal para ampliação de leitos em hospitais regionais.",
#     "Construir uma ponte ligando o Brasil ao Japão até o fim do mandato."
# ]

# for ex in exemplos:
#     resultado = classificar_proposta(ex)
#     print(f"Proposta: {ex}\nClassificação: {resultado}\n{'-'*50}")  

In [ ]:
gguf_directory = "modelo_propostas_gguf"
quantization_method = "q4_k_m" # Opções: "q4_k_m", "q8_0", "f16"
# pra rodar localmente ou em outras plataformas
print(f"Exportando modelo para GGUF em quantização {quantization_method}...")

model.save_pretrained_gguf(
    gguf_directory, 
    tokenizer, 
    quantization_method = quantization_method
)

print(f"Modelo GGUF salvo com sucesso na pasta: ./{gguf_directory}")